# Fine-tuning Exploration – Multilingual JUDGEBERT

This notebook walks through the key design choices for fine-tuning the LMP regression model.

**Architecture (same spirit as original JUDGEBERT / MeaningBERT):**
- Base encoder: `xlm-roberta-base` (multilingual)
- Input format: `[CLS] original [SEP] simplified [SEP]`
- Head: small MLP on the `[CLS]` representation → scalar LMP score ∈ [1, 10]
- Loss: Smooth L1 (Huber)

**Key fine-tuning practices used:**
1. Layer-wise learning rates (higher LR for the new head)
2. Optional JUDGEBERT-DA style augmentation (identical + unrelated pairs)
3. Early stopping on validation RMSE
4. Gradient clipping + optional mixed precision

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import torch
from model import JudgeBERTRegressor, get_tokenizer
from utils import load_dataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Dataset overview

In [ ]:
df = load_dataset("../data/lmp_dataset.jsonl")
print(df.shape)
display(df.head(3))
print("\nLanguage × Quality:")
print(df.groupby(["language", "quality"]).size().unstack(fill_value=0))
print("\nLMP score distribution:")
df.lmp_score.hist(bins=20)

## 2. Model architecture

In [ ]:
model = JudgeBERTRegressor("xlm-roberta-base", dropout=0.1)
print(model)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

## 3. Recommended fine-tuning commands

### Baseline (good starting point)
```bash
python src/train.py \
  --model_name xlm-roberta-base \
  --epochs 8 \
  --batch_size 8 \
  --lr 2e-5 \
  --head_lr_mult 10 \
  --augment \
  --patience 3 \
  --output_dir models/judgebert_xlmr
```

### Stronger / more stable (if you have a GPU)
```bash
python src/train.py \
  --model_name xlm-roberta-base \
  --epochs 12 \
  --batch_size 16 \
  --lr 1e-5 \
  --head_lr_mult 10 \
  --grad_accum 2 \
  --fp16 \
  --scheduler cosine \
  --augment \
  --patience 4 \
  --output_dir models/judgebert_xlmr_strong
```

### Train head only (fast experiment)
```bash
python src/train.py --freeze_base --epochs 5 --lr 1e-3 --output_dir models/head_only
```

## 4. What the original JUDGEBERT paper did

- Base model: **CamemBERT** (French)
- Task: regression on concatenated sentence pair
- 10-fold cross-validation
- Two variants:
  - JUDGEBERT (trained only on the 297 annotated pairs)
  - **JUDGEBERT-DA** (augmented with identical + unrelated pairs) → much better sanity-check performance

Our script implements the **DA** idea with the `--augment` flag (enabled by default).

## 5. After training – evaluation & qualitative analysis

```bash
python src/evaluate.py --checkpoint models/judgebert_xlmr/best_model.pt
python src/explain.py
```